# Tissue Ontology Coverage Analysis

This notebook analyzes:
1. Which tissue ontology IDs from training/test data are in CellxGene's curated vocabulary
2. Which tissues are missing from the vocabulary
3. Whether the TissueEncoder is loading the ontology correctly
4. Impact on training and test data filtering

In [20]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from collections import Counter
from tqdm import tqdm

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from data_loading.tissue_encoder import TissueEncoder

## 1. Load TissueEncoder and Inspect Vocabulary

In [21]:
# Initialize encoder
encoder = TissueEncoder()

print("TissueEncoder Vocabulary:")
print(f"  Tissues: {len(encoder.tissue_to_idx)}")
print(f"  Organs: {len(encoder.organ_to_idx)}")
print(f"  Systems: {len(encoder.system_to_idx)}")
print(f"  Total dimensions: {encoder.total_dim}")

# Show sample tissues
print("\nSample tissues (first 20):")
for i, tissue_id in enumerate(sorted(encoder.tissue_to_idx.keys())[:20]):
    print(f"  {tissue_id}")

TissueEncoder Vocabulary:
  Tissues: 81
  Organs: 28
  Systems: 17
  Total dimensions: 126

Sample tissues (first 20):
  UBERON:0000004
  UBERON:0000010
  UBERON:0000014
  UBERON:0000029
  UBERON:0000030
  UBERON:0000056
  UBERON:0000057
  UBERON:0000059
  UBERON:0000160
  UBERON:0000175
  UBERON:0000178
  UBERON:0000310
  UBERON:0000344
  UBERON:0000383
  UBERON:0000473
  UBERON:0000916
  UBERON:0000922
  UBERON:0000945
  UBERON:0000948
  UBERON:0000949


## 2. Verify Ontology Loading

Check if the ontology parser and curated lists are being loaded correctly.

In [22]:
# Check what's in the curated lists
print("Curated Lists from CellxGene:")
print(f"  curated_tissues: {len(encoder.curated_tissues)} terms")
print(f"  curated_organs: {len(encoder.curated_organs)} terms")
print(f"  curated_systems: {len(encoder.curated_systems)} terms")

# Show samples
print("\nSample curated tissues:")
for tissue in list(encoder.curated_tissues)[:10]:
    print(f"  {tissue}")

print("\nSample curated organs:")
for organ in list(encoder.curated_organs)[:10]:
    print(f"  {organ}")

print("\nSample curated systems:")
for system in list(encoder.curated_systems)[:10]:
    print(f"  {system}")

Curated Lists from CellxGene:
  curated_tissues: 81 terms
  curated_organs: 28 terms
  curated_systems: 17 terms

Sample curated tissues:
  UBERON:0000178
  UBERON:0002048
  UBERON:0002106
  UBERON:0002371
  UBERON:0002107
  UBERON:0002113
  UBERON:0000955
  UBERON:0002240
  UBERON:0000310
  UBERON:0000948

Sample curated organs:
  UBERON:0000992
  UBERON:0000029
  UBERON:0002048
  UBERON:0002110
  UBERON:0001043
  UBERON:0003889
  UBERON:0018707
  UBERON:0000178
  UBERON:0002371
  UBERON:0000955

Sample curated systems:
  UBERON:0001017
  UBERON:0004535
  UBERON:0001009
  UBERON:0001007
  UBERON:0000922
  UBERON:0000949
  UBERON:0002330
  UBERON:0002390
  UBERON:0002405
  UBERON:0000383


In [23]:
# Check ontology parser
print("Ontology Parser:")
print(f"  Type: {type(encoder.ontology_parser)}")
print(f"  Has ontology data: {hasattr(encoder.ontology_parser, 'ontology')}")

# Test with a known tissue ID
test_id = 'UBERON:0002107'  # liver
print(f"\nTest lookup for {test_id} (liver):")
print(f"  In tissue_to_idx: {test_id in encoder.tissue_to_idx}")
if test_id in encoder.tissue_to_idx:
    print(f"  Index: {encoder.tissue_to_idx[test_id]}")
    
    # Check hierarchy
    if test_id in encoder.tissue_to_organ_indices:
        organ_indices = encoder.tissue_to_organ_indices[test_id]
        print(f"  Organ indices: {organ_indices}")
    if test_id in encoder.tissue_to_system_indices:
        system_indices = encoder.tissue_to_system_indices[test_id]
        print(f"  System indices: {system_indices}")

Ontology Parser:
  Type: <class 'cellxgene_ontology_guide.ontology_parser.OntologyParser'>
  Has ontology data: False

Test lookup for UBERON:0002107 (liver):
  In tissue_to_idx: True
  Index: 54
  Organ indices: [20]
  System indices: [3, 6, 13]


## 3. Analyze Training Data

In [24]:
# Load training data tissue ontology IDs
training_dir = Path('/mmc-scratch/scratch/cellxgene_v2_training_v1')
training_files = sorted(training_dir.glob('*.parquet'))

print(f"Found {len(training_files)} training files")
print("Sampling tissue ontology IDs from training data (first 100 files)...")

training_tissue_counts = Counter()
training_total_cells = 0

for pq_file in tqdm(training_files[:100], desc="Processing training files"):
    df = pd.read_parquet(pq_file, columns=['tissue_ontology_term_id'])
    training_tissue_counts.update(df['tissue_ontology_term_id'].value_counts().to_dict())
    training_total_cells += len(df)

training_tissues = set(training_tissue_counts.keys())
print(f"\nTraining data (sampled):")
print(f"  Total cells: {training_total_cells:,}")
print(f"  Unique tissue ontology IDs: {len(training_tissues)}")

Found 643 training files
Sampling tissue ontology IDs from training data (first 100 files)...


Processing training files:   0%|          | 0/100 [00:00<?, ?it/s]

Processing training files: 100%|██████████| 100/100 [00:09<00:00, 10.13it/s]


Training data (sampled):
  Total cells: 674,026
  Unique tissue ontology IDs: 136


In [25]:
# Check coverage
encoder_tissues = set(encoder.tissue_to_idx.keys())

training_in_encoder = training_tissues & encoder_tissues
training_not_in_encoder = training_tissues - encoder_tissues

print("Training Data Coverage:")
print(f"  Tissues in encoder: {len(training_in_encoder)} / {len(training_tissues)} ({len(training_in_encoder)/len(training_tissues)*100:.1f}%)")
print(f"  Tissues NOT in encoder: {len(training_not_in_encoder)} ({len(training_not_in_encoder)/len(training_tissues)*100:.1f}%)")

# Count cells affected
cells_with_valid_tissue = sum(training_tissue_counts[t] for t in training_in_encoder)
cells_with_invalid_tissue = sum(training_tissue_counts[t] for t in training_not_in_encoder)

print(f"\nCell-level impact:")
print(f"  Cells with valid tissue: {cells_with_valid_tissue:,} ({cells_with_valid_tissue/training_total_cells*100:.1f}%)")
print(f"  Cells with invalid tissue: {cells_with_invalid_tissue:,} ({cells_with_invalid_tissue/training_total_cells*100:.1f}%)")

Training Data Coverage:
  Tissues in encoder: 26 / 136 (19.1%)
  Tissues NOT in encoder: 110 (80.9%)

Cell-level impact:
  Cells with valid tissue: 221,363 (32.8%)
  Cells with invalid tissue: 452,663 (67.2%)


In [26]:
# Show missing tissues from training
if len(training_not_in_encoder) > 0:
    print("\nTop 20 missing tissues in TRAINING (by cell count):")
    missing_tissues_sorted = sorted(
        [(t, training_tissue_counts[t]) for t in training_not_in_encoder],
        key=lambda x: x[1],
        reverse=True
    )
    for tissue_id, count in missing_tissues_sorted[:20]:
        print(f"  {tissue_id:30s} {count:8,} cells")
else:
    print("\n✓ All training tissues are in encoder vocabulary!")


Top 20 missing tissues in TRAINING (by cell count):
  UBERON:0001225                   60,000 cells
  UBERON:0008946                   53,042 cells
  UBERON:0001005                   31,435 cells
  UBERON:0002084                   30,666 cells
  UBERON:0000991                   30,048 cells
  UBERON:0000362                   26,142 cells
  CL:0000084                       25,229 cells
  UBERON:0001775                   21,997 cells
  CL:0002328                       19,164 cells
  UBERON:0012648                   11,008 cells
  UBERON:8410010                    9,351 cells
  UBERON:0009834                    9,125 cells
  UBERON:0001786                    8,243 cells
  CL:0002335                        7,594 cells
  UBERON:0001228                    7,301 cells
  UBERON:0001950                    6,068 cells
  UBERON:0001111                    5,568 cells
  UBERON:0000451                    4,926 cells
  UBERON:0013756                    4,733 cells
  UBERON:0008953                   

## 4. Analyze Test Data

In [27]:
# Load test data tissue ontology IDs
test_dir = Path('/mmc-scratch/scratch/cellxgene_v2_test_v1')
test_files = sorted(test_dir.glob('*.parquet'))

print(f"Found {len(test_files)} test files")
print("Loading tissue ontology IDs from test data...")

test_tissue_counts = Counter()
test_total_cells = 0

for pq_file in tqdm(test_files, desc="Processing test files"):
    df = pd.read_parquet(pq_file, columns=['tissue_ontology_term_id'])
    test_tissue_counts.update(df['tissue_ontology_term_id'].value_counts().to_dict())
    test_total_cells += len(df)

test_tissues = set(test_tissue_counts.keys())
print(f"\nTest data:")
print(f"  Total cells: {test_total_cells:,}")
print(f"  Unique tissue ontology IDs: {len(test_tissues)}")

Found 15 test files
Loading tissue ontology IDs from test data...


Processing test files: 100%|██████████| 15/15 [00:01<00:00, 11.48it/s]


Test data:
  Total cells: 120,984
  Unique tissue ontology IDs: 17


In [28]:
# Check coverage
test_in_encoder = test_tissues & encoder_tissues
test_not_in_encoder = test_tissues - encoder_tissues

print("Test Data Coverage:")
print(f"  Tissues in encoder: {len(test_in_encoder)} / {len(test_tissues)} ({len(test_in_encoder)/len(test_tissues)*100:.1f}%)")
print(f"  Tissues NOT in encoder: {len(test_not_in_encoder)} ({len(test_not_in_encoder)/len(test_tissues)*100:.1f}%)")

# Count cells affected
cells_with_valid_tissue = sum(test_tissue_counts[t] for t in test_in_encoder)
cells_with_invalid_tissue = sum(test_tissue_counts[t] for t in test_not_in_encoder)

print(f"\nCell-level impact:")
print(f"  Cells with valid tissue: {cells_with_valid_tissue:,} ({cells_with_valid_tissue/test_total_cells*100:.1f}%)")
print(f"  Cells with invalid tissue: {cells_with_invalid_tissue:,} ({cells_with_invalid_tissue/test_total_cells*100:.1f}%)")

Test Data Coverage:
  Tissues in encoder: 5 / 17 (29.4%)
  Tissues NOT in encoder: 12 (70.6%)

Cell-level impact:
  Cells with valid tissue: 50,206 (41.5%)
  Cells with invalid tissue: 70,778 (58.5%)


In [29]:
# Show missing tissues from test
if len(test_not_in_encoder) > 0:
    print("\nMissing tissues in TEST (by cell count):")
    missing_tissues_sorted = sorted(
        [(t, test_tissue_counts[t]) for t in test_not_in_encoder],
        key=lambda x: x[1],
        reverse=True
    )
    for tissue_id, count in missing_tissues_sorted:
        # Try to get tissue name
        sample_file = next(test_dir.glob('*.parquet'))
        df_sample = pd.read_parquet(sample_file, columns=['tissue_ontology_term_id', 'tissue'])
        name_row = df_sample[df_sample['tissue_ontology_term_id'] == tissue_id]
        if len(name_row) > 0:
            tissue_name = name_row['tissue'].iloc[0]
        else:
            tissue_name = "(name not found)"
        
        print(f"  {tissue_id:30s} {count:8,} cells - {tissue_name}")
else:
    print("\n✓ All test tissues are in encoder vocabulary!")


Missing tissues in TEST (by cell count):
  UBERON:0002686                   22,033 cells - (name not found)
  UBERON:0001111                   11,806 cells - (name not found)
  UBERON:8480009                    8,991 cells - tendon of semitendinosus
  UBERON:0014614                    5,347 cells - (name not found)
  UBERON:0002728                    4,991 cells - (name not found)
  UBERON:0002317                    4,974 cells - (name not found)
  UBERON:0013535                    3,364 cells - (name not found)
  UBERON:0002185                    3,103 cells - (name not found)
  UBERON:0008952                    2,170 cells - (name not found)
  UBERON:0008953                    1,765 cells - (name not found)
  UBERON:0003544                    1,517 cells - (name not found)
  UBERON:0003126                      717 cells - (name not found)


## 5. Compare Training vs Test Coverage

In [30]:
# Venn diagram style comparison
only_in_training = training_not_in_encoder - test_not_in_encoder
only_in_test = test_not_in_encoder - training_not_in_encoder
missing_in_both = training_not_in_encoder & test_not_in_encoder

print("Missing Tissue Comparison:")
print(f"  Missing ONLY in training: {len(only_in_training)}")
print(f"  Missing ONLY in test: {len(only_in_test)}")
print(f"  Missing in BOTH: {len(missing_in_both)}")

if len(only_in_test) > 0:
    print("\nTissues missing ONLY in test (problematic for evaluation):")
    for tissue_id in sorted(only_in_test):
        count = test_tissue_counts[tissue_id]
        print(f"  {tissue_id:30s} {count:8,} cells")

Missing Tissue Comparison:
  Missing ONLY in training: 104
  Missing ONLY in test: 6
  Missing in BOTH: 6

Tissues missing ONLY in test (problematic for evaluation):
  UBERON:0002185                    3,103 cells
  UBERON:0002686                   22,033 cells
  UBERON:0003126                      717 cells
  UBERON:0003544                    1,517 cells
  UBERON:0008952                    2,170 cells
  UBERON:8480009                    8,991 cells


## 6. Summary and Recommendations

In [31]:
print("=" * 100)
print("SUMMARY")
print("=" * 100)

print("\nEncoder Vocabulary:")
print(f"  Total tissues: {len(encoder_tissues)}")
print(f"  Dimensions: {encoder.total_dim} (81 tissue + 28 organ + 17 system)")

print("\nTraining Data (sampled):")
print(f"  Unique tissues: {len(training_tissues)}")
print(f"  Coverage: {len(training_in_encoder)}/{len(training_tissues)} ({len(training_in_encoder)/len(training_tissues)*100:.1f}%)")
print(f"  Cells with valid tissue: {cells_with_valid_tissue:,} ({cells_with_valid_tissue/training_total_cells*100:.1f}%)")

print("\nTest Data:")
test_valid = sum(test_tissue_counts[t] for t in test_in_encoder)
test_invalid = sum(test_tissue_counts[t] for t in test_not_in_encoder)
print(f"  Unique tissues: {len(test_tissues)}")
print(f"  Coverage: {len(test_in_encoder)}/{len(test_tissues)} ({len(test_in_encoder)/len(test_tissues)*100:.1f}%)")
print(f"  Cells with valid tissue: {test_valid:,} ({test_valid/test_total_cells*100:.1f}%)")
print(f"  Cells with invalid tissue: {test_invalid:,} ({test_invalid/test_total_cells*100:.1f}%)")

print("\n" + "=" * 100)
print("RECOMMENDATIONS")
print("=" * 100)

if test_invalid > 0:
    pct_invalid = test_invalid / test_total_cells * 100
    print(f"\n⚠️  {pct_invalid:.1f}% of test cells have tissues not in the encoder vocabulary")
    print("\nOptions:")
    print("  A. Filter these cells during evaluation (track_invalid_embeddings=True)")
    print("     - Ensures model sees consistent data distribution")
    print(f"     - Evaluate on {test_valid:,} cells ({test_valid/test_total_cells*100:.1f}%)")
    print("\n  B. Don't filter (track_invalid_embeddings=False)")
    print("     - Keep all cells but with zero tissue vectors for unknown tissues")
    print(f"     - Evaluate on {test_total_cells:,} cells (100%)")
    print("     - May degrade performance due to missing features")
    print("\n  C. Remove tissue embeddings entirely")
    print("     - Change embedding_types to ['genept', 'metadata']")
    print("     - Simpler but loses tissue information")
    print("\n  D. Expand encoder vocabulary to include missing tissues")
    print("     - Requires modifying TissueEncoder or CellxGene curated lists")
    print("     - Most complete but requires development work")
else:
    print("\n✓ All test tissues are covered by the encoder vocabulary!")
    print("  No action needed for tissue encoding.")

SUMMARY

Encoder Vocabulary:
  Total tissues: 81
  Dimensions: 126 (81 tissue + 28 organ + 17 system)

Training Data (sampled):
  Unique tissues: 136
  Coverage: 26/136 (19.1%)
  Cells with valid tissue: 50,206 (7.4%)

Test Data:
  Unique tissues: 17
  Coverage: 5/17 (29.4%)
  Cells with valid tissue: 50,206 (41.5%)
  Cells with invalid tissue: 70,778 (58.5%)

RECOMMENDATIONS

⚠️  58.5% of test cells have tissues not in the encoder vocabulary

Options:
  A. Filter these cells during evaluation (track_invalid_embeddings=True)
     - Ensures model sees consistent data distribution
     - Evaluate on 50,206 cells (41.5%)

  B. Don't filter (track_invalid_embeddings=False)
     - Keep all cells but with zero tissue vectors for unknown tissues
     - Evaluate on 120,984 cells (100%)
     - May degrade performance due to missing features

  C. Remove tissue embeddings entirely
     - Change embedding_types to ['genept', 'metadata']
     - Simpler but loses tissue information

  D. Expand enc

In [32]:
# Create DataFrame of missing test tissues with names
if len(test_not_in_encoder) > 0:
    missing_data = []
    
    for pq_file in test_files:
        df = pd.read_parquet(pq_file, columns=['tissue_ontology_term_id', 'tissue'])
        for tissue_id in test_not_in_encoder:
            matches = df[df['tissue_ontology_term_id'] == tissue_id]
            if len(matches) > 0:
                tissue_name = matches['tissue'].iloc[0]
                if not any(d['tissue_id'] == tissue_id for d in missing_data):
                    missing_data.append({
                        'tissue_id': tissue_id,
                        'tissue_name': tissue_name,
                        'test_cells': test_tissue_counts[tissue_id]
                    })
    
    missing_df = pd.DataFrame(missing_data).sort_values('test_cells', ascending=False)
    
    output_file = Path('../analysis/missing_tissues.csv')
    output_file.parent.mkdir(exist_ok=True)
    missing_df.to_csv(output_file, index=False)
    
    print(f"Exported missing tissues to: {output_file}")
    print(f"\nPreview:")
    print(missing_df.to_string(index=False))

Exported missing tissues to: ../analysis/missing_tissues.csv

Preview:
     tissue_id                       tissue_name  test_cells
UBERON:0002686                     angular gyrus       22033
UBERON:0001111                intercostal muscle       11806
UBERON:8480009          tendon of semitendinosus        8991
UBERON:0014614 cervical spinal cord white matter        5347
UBERON:0002728                 entorhinal cortex        4991
UBERON:0002317        white matter of cerebellum        4974
UBERON:0013535            Brodmann (1909) area 4        3364
UBERON:0002185                          bronchus        3103
UBERON:0008952           upper lobe of left lung        2170
UBERON:0008953           lower lobe of left lung        1765
UBERON:0003544                brain white matter        1517
UBERON:0003126                           trachea         717


In [33]:
# Export all candidate data to CSV for manual review
output_dir = Path('../analysis')
output_dir.mkdir(exist_ok=True)

# 1. Export candidate organs/systems
candidates_df = pd.DataFrame([{
    'uberon_id': c['tissue_id'],
    'label': c['label'],
    'category': c['category'],
    'num_descendant_tissues': c['num_descendants'],
    'total_cells_covered': c['total_cells_covered'],
    'example_tissues': '|'.join(c['example_tissues'][:5])
} for c in candidate_hierarchy_info])

candidates_file = output_dir / 'candidate_organs_systems.csv'
candidates_df.to_csv(candidates_file, index=False)
print(f"Exported {len(candidates_df)} candidates to: {candidates_file}")

# 2. Export tissues with NO mapping
no_mapping_df = pd.DataFrame([{
    'uberon_id': t['tissue_id'],
    'training_cells': t['training_cells'],
    'test_cells': t['test_cells'],
    'total_cells': t['total_cells']
} for t in sorted_no_mapping[:20]])

no_mapping_file = output_dir / 'tissues_with_no_mapping.csv'
no_mapping_df.to_csv(no_mapping_file, index=False)
print(f"Exported {len(no_mapping_df)} tissues with no mapping to: {no_mapping_file}")

# 3. Export mapping statistics summary
mapping_summary = {
    'Total missing tissues': len(all_missing_tissues),
    'Tissues with no mapping': len(tissues_with_no_mapping),
    'Tissues with organ only': len(tissues_with_organ_only),
    'Tissues with system only': len(tissues_with_system_only),
    'Tissues with both organ and system': len(tissues_with_both),
    'Total cells in missing tissues': sum(t['total_cells'] for t in tissues_with_no_mapping + tissues_with_organ_only + tissues_with_system_only + tissues_with_both),
    'Cells with no mapping': sum(t['total_cells'] for t in tissues_with_no_mapping)
}

summary_df = pd.DataFrame([mapping_summary]).T
summary_df.columns = ['Count']
summary_file = output_dir / 'mapping_summary.csv'
summary_df.to_csv(summary_file)
print(f"Exported mapping summary to: {summary_file}")

print("\n" + "=" * 100)
print("CURATION REPORT COMPLETE")
print("=" * 100)
print(f"\nReview these files to decide which candidates to add to curated lists:")
print(f"  1. {candidates_file}")
print(f"  2. {no_mapping_file}")
print(f"  3. {summary_file}")
print(f"\nNext steps:")
print(f"  - Review system-like candidates (20+ descendants) for addition to curated_systems")
print(f"  - Review organ-like candidates (5-19 descendants) for addition to curated_organs")
print(f"  - Investigate tissues with no mapping to determine if they need special handling")

NameError: name 'candidate_hierarchy_info' is not defined

## 11. Export Curation Report

Export comprehensive data for manual curation review.

In [ ]:
# Categorize candidates by their relationship to existing curated terms
# Heuristic: check how many descendants each candidate has
print("\n" + "=" * 100)
print("CATEGORIZING CANDIDATES")
print("=" * 100)
print("Estimating hierarchy level (organ-like vs system-like) for top candidates...")

candidate_hierarchy_info = []

for candidate in sorted_by_coverage[:30]:
    candidate_id = candidate['tissue_id']
    
    # Count how many of our missing tissues are descendants
    num_descendants = candidate['covered_tissues']
    
    # Rough heuristic:
    # - Systems typically have many descendants (>20)
    # - Organs typically have moderate descendants (5-20)
    # - Tissues typically have few descendants (<5)
    
    if num_descendants >= 20:
        category = "System-like"
    elif num_descendants >= 5:
        category = "Organ-like"
    else:
        category = "Tissue-like"
    
    candidate_hierarchy_info.append({
        **candidate,
        'category': category,
        'num_descendants': num_descendants
    })

# Group by category
print("\nSYSTEM-LIKE CANDIDATES (20+ descendant tissues):")
print(f"{'UBERON ID':<20} {'Label':<40} {'Descendants':<12} {'Cells':<12}")
print("-" * 100)
for c in candidate_hierarchy_info:
    if c['category'] == "System-like":
        print(f"{c['tissue_id']:<20} {c['label'][:38]:<40} {c['num_descendants']:<12} {c['total_cells_covered']:>11,}")

print("\n\nORGAN-LIKE CANDIDATES (5-19 descendant tissues):")
print(f"{'UBERON ID':<20} {'Label':<40} {'Descendants':<12} {'Cells':<12}")
print("-" * 100)
for c in candidate_hierarchy_info:
    if c['category'] == "Organ-like":
        print(f"{c['tissue_id']:<20} {c['label'][:38]:<40} {c['num_descendants']:<12} {c['total_cells_covered']:>11,}")

print("\n\nTISSUE-LIKE CANDIDATES (1-4 descendant tissues):")
print(f"{'UBERON ID':<20} {'Label':<40} {'Descendants':<12} {'Cells':<12}")
print("-" * 100)
for c in candidate_hierarchy_info:
    if c['category'] == "Tissue-like":
        print(f"{c['tissue_id']:<20} {c['label'][:38]:<40} {c['num_descendants']:<12} {c['total_cells_covered']:>11,}")

## 10. Categorize Candidates by Hierarchy Level

Determine which candidates are organ-like vs system-like based on their position in the ontology.

In [ ]:
# For each candidate, calculate total cell count coverage
print("\n" + "=" * 100)
print("CANDIDATE COVERAGE ANALYSIS")
print("=" * 100)
print("Calculating how many cells each candidate would cover if added to curated lists...")

# Map candidates to tissues that would benefit
candidate_to_tissues = {cand['tissue_id']: [] for cand in sorted_candidates[:50]}

for tissue_id in all_missing_tissues:
    try:
        ancestors = encoder.ontology_parser.get_term_ancestors(tissue_id, include_self=True)
        
        # Check which top candidates are ancestors
        for candidate_id in list(candidate_to_tissues.keys()):
            if candidate_id in ancestors:
                training_count = training_tissue_counts.get(tissue_id, 0)
                test_count = test_tissue_counts.get(tissue_id, 0)
                candidate_to_tissues[candidate_id].append({
                    'tissue_id': tissue_id,
                    'training_cells': training_count,
                    'test_cells': test_count,
                    'total_cells': training_count + test_count
                })
    except:
        pass

# Add coverage info to candidates
for candidate in sorted_candidates[:50]:
    tissues = candidate_to_tissues[candidate['tissue_id']]
    total_coverage = sum(t['total_cells'] for t in tissues)
    candidate['covered_tissues'] = len(tissues)
    candidate['total_cells_covered'] = total_coverage
    candidate['example_tissues'] = [t['tissue_id'] for t in tissues[:5]]

# Sort by total cell coverage
sorted_by_coverage = sorted(sorted_candidates[:50], key=lambda x: x['total_cells_covered'], reverse=True)

print("\nTop 30 candidates by cell coverage:")
print(f"{'Rank':<5} {'UBERON ID':<20} {'Label':<40} {'Tissues':<10} {'Cells':<12}")
print("-" * 100)

for i, candidate in enumerate(sorted_by_coverage[:30], 1):
    print(f"{i:<5} {candidate['tissue_id']:<20} {candidate['label'][:38]:<40} "
          f"{candidate['covered_tissues']:<10} {candidate['total_cells_covered']:>11,}")

In [ ]:
# Find candidates: ancestors that are NOT in curated lists but appear frequently
print("Finding candidate organs/systems for curation...")
print("=" * 100)

# Get all curated terms
all_curated = encoder.curated_tissues | encoder.curated_organs_set | encoder.curated_systems_set

# Find candidate ancestors (not in curated lists, but appear in ancestry)
candidate_ancestors = {}
for ancestor_id, frequency in all_ancestors_counter.items():
    if ancestor_id not in all_curated:
        # Get label for this term
        try:
            label = encoder.ontology_parser.get_term_label(ancestor_id)
        except:
            label = "(unknown)"
        
        candidate_ancestors[ancestor_id] = {
            'label': label,
            'frequency': frequency,
            'tissue_id': ancestor_id
        }

print(f"Found {len(candidate_ancestors):,} candidate ancestors not in curated lists")
print(f"Total curated terms: {len(all_curated)}")
print(f"Total ancestors encountered: {len(all_ancestors_counter)}")

# Sort by frequency (how many missing tissues have this as an ancestor)
sorted_candidates = sorted(candidate_ancestors.values(), key=lambda x: x['frequency'], reverse=True)

print("\nTop 50 candidate ancestors by frequency:")
print(f"{'Rank':<5} {'UBERON ID':<20} {'Label':<50} {'Frequency':<10}")
print("-" * 100)

for i, candidate in enumerate(sorted_candidates[:50], 1):
    print(f"{i:<5} {candidate['tissue_id']:<20} {candidate['label'][:48]:<50} {candidate['frequency']:<10}")

## 9. Identify Candidate Organs/Systems for Curation

Find frequently-occurring ancestors that are NOT in curated lists but could be added.

In [ ]:
# Show details of tissues with NO mapping
if tissues_with_no_mapping:
    print("\n" + "=" * 100)
    print("CRITICAL: TISSUES WITH NO ORGAN/SYSTEM MAPPING")
    print("=" * 100)
    print("These tissues cannot map to ANY curated organ or system via ancestry.")
    print("Options:")
    print("  1. Add ancestors of these tissues to curated lists")
    print("  2. Filter these tissues from training/test data")
    print("  3. Investigate if these are data quality issues (e.g., CL terms instead of UBERON)")
    print("\nTop 20 by cell count:")
    
    # Sort by total cells
    sorted_no_mapping = sorted(tissues_with_no_mapping, key=lambda x: x['total_cells'], reverse=True)
    
    for i, tissue in enumerate(sorted_no_mapping[:20], 1):
        print(f"{i:2d}. {tissue['tissue_id']:20s} "
              f"Training: {tissue['training_cells']:8,} | "
              f"Test: {tissue['test_cells']:8,} | "
              f"Total: {tissue['total_cells']:8,}")
        
    # Calculate impact
    total_cells_no_mapping = sum(t['total_cells'] for t in tissues_with_no_mapping)
    total_missing_cells = sum(training_tissue_counts[t] for t in all_missing_tissues) + sum(test_tissue_counts[t] for t in all_missing_tissues)
    print(f"\nTotal cells affected: {total_cells_no_mapping:,}")
    print(f"Percentage of all missing tissue cells: {total_cells_no_mapping / total_missing_cells * 100:.1f}%")
else:
    print("\n✓ All missing tissues can map to at least one curated organ or system!")

In [ ]:
# For each missing tissue, check if it maps to ANY curated organ/system via ancestors
print("Analyzing ancestor mappings for missing tissues...")
print("=" * 100)

# Combine all missing tissues from training and test
all_missing_tissues = training_not_in_encoder | test_not_in_encoder

# Track mapping results
tissues_with_no_mapping = []
tissues_with_organ_only = []
tissues_with_system_only = []
tissues_with_both = []

# Track all ancestors encountered (for candidate identification)
all_ancestors_counter = Counter()

for tissue_id in tqdm(sorted(all_missing_tissues), desc="Checking ancestor mappings"):
    try:
        # Get all ancestors for this tissue
        ancestors = encoder.ontology_parser.get_term_ancestors(tissue_id, include_self=True)
        
        # Track ALL ancestors for frequency analysis
        all_ancestors_counter.update(ancestors)
        
        # Find which curated organs/systems are in ancestors
        matching_organs = [org for org in ancestors if org in encoder.curated_organs_set]
        matching_systems = [sys for sys in ancestors if sys in encoder.curated_systems_set]
        
        # Get cell counts
        training_count = training_tissue_counts.get(tissue_id, 0)
        test_count = test_tissue_counts.get(tissue_id, 0)
        total_count = training_count + test_count
        
        tissue_info = {
            'tissue_id': tissue_id,
            'training_cells': training_count,
            'test_cells': test_count,
            'total_cells': total_count,
            'organ_ancestors': matching_organs,
            'system_ancestors': matching_systems
        }
        
        # Categorize based on mappings
        if not matching_organs and not matching_systems:
            tissues_with_no_mapping.append(tissue_info)
        elif matching_organs and not matching_systems:
            tissues_with_organ_only.append(tissue_info)
        elif not matching_organs and matching_systems:
            tissues_with_system_only.append(tissue_info)
        else:
            tissues_with_both.append(tissue_info)
            
    except Exception as e:
        print(f"Error processing {tissue_id}: {e}")
        tissues_with_no_mapping.append({
            'tissue_id': tissue_id,
            'training_cells': training_tissue_counts.get(tissue_id, 0),
            'test_cells': test_tissue_counts.get(tissue_id, 0),
            'total_cells': training_tissue_counts.get(tissue_id, 0) + test_tissue_counts.get(tissue_id, 0),
            'organ_ancestors': [],
            'system_ancestors': []
        })

print("\n" + "=" * 100)
print("MAPPING RESULTS")
print("=" * 100)
print(f"  Tissues with NO mapping (neither organ nor system): {len(tissues_with_no_mapping)}")
print(f"  Tissues with organ-only mapping: {len(tissues_with_organ_only)}")
print(f"  Tissues with system-only mapping: {len(tissues_with_system_only)}")
print(f"  Tissues with both organ and system mapping: {len(tissues_with_both)}")
print(f"\nTotal missing tissues analyzed: {len(all_missing_tissues)}")

## 8. Find Tissues with No Organ/System Mapping

Analyze which tissues have NO mapping to curated organs or systems via ancestry.

In [17]:
# For each candidate, calculate total cell count coverage
print("\n" + "=" * 100)
print("CANDIDATE COVERAGE ANALYSIS")
print("=" * 100)
print("Calculating how many cells each candidate would cover if added to curated lists...")

# Map candidates to tissues that would benefit
candidate_to_tissues = {cand['tissue_id']: [] for cand in sorted_candidates[:50]}

for tissue_id in all_missing_tissues:
    try:
        ancestors = encoder.ontology_parser.get_term_ancestors(tissue_id, include_self=True)
        
        # Check which top candidates are ancestors
        for candidate_id in list(candidate_to_tissues.keys()):
            if candidate_id in ancestors:
                training_count = training_tissue_counts.get(tissue_id, 0)
                test_count = test_tissue_counts.get(tissue_id, 0)
                candidate_to_tissues[candidate_id].append({
                    'tissue_id': tissue_id,
                    'training_cells': training_count,
                    'test_cells': test_count,
                    'total_cells': training_count + test_count
                })
    except:
        pass

# Add coverage info to candidates
for candidate in sorted_candidates[:50]:
    tissues = candidate_to_tissues[candidate['tissue_id']]
    total_coverage = sum(t['total_cells'] for t in tissues)
    candidate['covered_tissues'] = len(tissues)
    candidate['total_cells_covered'] = total_coverage
    candidate['example_tissues'] = [t['tissue_id'] for t in tissues[:5]]

# Sort by total cell coverage
sorted_by_coverage = sorted(sorted_candidates[:50], key=lambda x: x['total_cells_covered'], reverse=True)

print("\nTop 30 candidates by cell coverage:")
print(f"{'Rank':<5} {'UBERON ID':<20} {'Label':<40} {'Tissues':<10} {'Cells':<12}")
print("-" * 100)

for i, candidate in enumerate(sorted_by_coverage[:30], 1):
    print(f"{i:<5} {candidate['tissue_id']:<20} {candidate['label'][:38]:<40} "
          f"{candidate['covered_tissues']:<10} {candidate['total_cells_covered']:>11,}")


CANDIDATE COVERAGE ANALYSIS
Calculating how many cells each candidate would cover if added to curated lists...


NameError: name 'sorted_candidates' is not defined

In [ ]:
# Export all candidate data to CSV for manual review
output_dir = Path('../analysis')
output_dir.mkdir(exist_ok=True)

# 1. Export candidate organs/systems
candidates_df = pd.DataFrame([{
    'uberon_id': c['tissue_id'],
    'label': c['label'],
    'category': c['category'],
    'num_descendant_tissues': c['num_descendants'],
    'total_cells_covered': c['total_cells_covered'],
    'example_tissues': '|'.join(c['example_tissues'][:5])
} for c in candidate_hierarchy_info])

candidates_file = output_dir / 'candidate_organs_systems.csv'
candidates_df.to_csv(candidates_file, index=False)
print(f"Exported {len(candidates_df)} candidates to: {candidates_file}")

# 2. Export tissues with NO mapping
no_mapping_df = pd.DataFrame([{
    'uberon_id': t['tissue_id'],
    'training_cells': t['training_cells'],
    'test_cells': t['test_cells'],
    'total_cells': t['total_cells']
} for t in sorted_no_mapping[:20]])

no_mapping_file = output_dir / 'tissues_with_no_mapping.csv'
no_mapping_df.to_csv(no_mapping_file, index=False)
print(f"Exported {len(no_mapping_df)} tissues with no mapping to: {no_mapping_file}")

# 3. Export mapping statistics summary
mapping_summary = {
    'Total missing tissues': len(all_missing_tissues),
    'Tissues with no mapping': len(tissues_with_no_mapping),
    'Tissues with organ only': len(tissues_with_organ_only),
    'Tissues with system only': len(tissues_with_system_only),
    'Tissues with both organ and system': len(tissues_with_both),
    'Total cells in missing tissues': sum(t['total_cells'] for t in tissues_with_no_mapping + tissues_with_organ_only + tissues_with_system_only + tissues_with_both),
    'Cells with no mapping': sum(t['total_cells'] for t in tissues_with_no_mapping)
}

summary_df = pd.DataFrame([mapping_summary]).T
summary_df.columns = ['Count']
summary_file = output_dir / 'mapping_summary.csv'
summary_df.to_csv(summary_file)
print(f"Exported mapping summary to: {summary_file}")

print("\n" + "=" * 100)
print("CURATION REPORT COMPLETE")
print("=" * 100)
print(f"\nReview these files to decide which candidates to add to curated lists:")
print(f"  1. {candidates_file}")
print(f"  2. {no_mapping_file}")
print(f"  3. {summary_file}")
print(f"\nNext steps:")
print(f"  - Review system-like candidates (20+ descendants) for addition to curated_systems")
print(f"  - Review organ-like candidates (5-19 descendants) for addition to curated_organs")
print(f"  - Investigate tissues with no mapping to determine if they need special handling")